# Housing Strand - Train Models

**MultimodalAI'26 - Housing Demo**

This notebook trains a single lightweight baseline model and writes artifacts needed for downstream evidence analysis.

**Objective**
- Load processed train/test features.
- Train one baseline model:
  - Model A: Logistic Regression
- Compute train and test metrics.
- Compute subgroup AUROC gaps relative to the reference group (`terraced`).
- Save model binary and metrics tables.

**Important starter-kit boundary**
- This notebook does **not** complete final deliverables.
- Participants must use outputs as evidence inputs and then fill root `reference/` templates manually.

**Prerequisite**
- `demo/data/processed/train_features.csv` and `test_features.csv` from Notebook 02.

**Sections**
1. Setup imports and output paths.
2. Load training/test matrices.
3. Train baseline and evaluate metrics.
4. Save outputs for Notebook 04 (Evidence Dashboard workbench).

## Section 1 - Setup and Artifact Paths

This cell imports model/evaluation/report utilities and prepares output directories:
- `saved_metrics/`
- `saved_models/`
- `reference/`

In [ ]:
from pathlib import Path
import sys
import json

import joblib
import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    DEMO_ROOT = cwd
elif (cwd / "demo" / "src").exists():
    DEMO_ROOT = cwd / "demo"
else:
    DEMO_ROOT = cwd.parent

sys.path.append(str(DEMO_ROOT))

from src.evaluate import compute_metrics, subgroup_auroc_gaps_reference, subgroup_auroc_scores
from src.models.model_a import build_model_a

PROCESSED_DIR = DEMO_ROOT / "data" / "processed"
METRICS_DIR = DEMO_ROOT / "saved_metrics"
MODELS_DIR = DEMO_ROOT / "saved_models"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## Section 2 - Load Processed Features

This cell loads train/test feature tables and defines the baseline feature list used for both models.

Participants can later expand this feature set with additional engineering.

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / "train_features.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test_features.csv")

features = [
    "avgTemperature",
    "avgHumidity",
    "co2_imputed",
    "co2_missing",
    "smart_meter_kwh",
    "noise_db",
    "lag_temp",
    "day_of_week",
    "is_flat",
]

X_train = train_df[features]
y_train = train_df["cold_risk"]
X_test = test_df[features]
y_test = test_df["cold_risk"]

## Section 3 - Train Baselines and Compute Metrics

This cell trains three demo models and computes key metrics on both train and test splits:
- AUROC, AUPRC, Brier score
- sensitivity/specificity/PPV/NPV/F1
- subgroup AUROC gaps by `property_type` relative to `terraced`

Outputs are saved to `saved_metrics/` and `saved_models/`.

In [ ]:
models = {
    "Model A Logistic Baseline": build_model_a(),
}

train_rows = []
test_rows = []
equity_rows = []
model_results = []
thresholds = [0.3, 0.5, 0.7]
threshold_counts = {str(t): 0 for t in thresholds}

for model_name, model in models.items():
    model.fit(X_train, y_train)

    y_prob_train = model.predict_proba(X_train)[:, 1]
    y_prob_test = model.predict_proba(X_test)[:, 1]

    train_metrics = compute_metrics(y_train, y_prob_train, threshold=0.5)
    train_metrics["n_properties"] = int(train_df["reference"].nunique())

    test_metrics = compute_metrics(y_test, y_prob_test, threshold=0.5)
    test_metrics["n_properties"] = int(test_df["reference"].nunique())

    subgroup_scores = subgroup_auroc_scores(test_df, y_prob_test, subgroup_col="property_type")
    subgroup_gaps = subgroup_auroc_gaps_reference(subgroup_scores, reference_group="terraced")

    train_rows.append({"model": model_name, "split": "train", **train_metrics})
    test_rows.append({"model": model_name, "split": "test", **test_metrics})
    model_results.append({"name": model_name, "metrics": test_metrics, "subgroup_gaps": subgroup_gaps})

    for t in thresholds:
        threshold_counts[str(t)] += int((y_prob_test >= t).sum())

    for group, score in subgroup_scores.items():
        equity_rows.append(
            {
                "model": model_name,
                "group": group,
                "auroc": 0.0 if pd.isna(score) else float(score),
                "gap_vs_terraced": float(subgroup_gaps.get(group, 0.0)),
            }
        )

    model_path = MODELS_DIR / (model_name.lower().replace(" ", "_").replace("-", "_") + ".joblib")
    joblib.dump(model, model_path)

train_metrics_df = pd.DataFrame(train_rows)
test_metrics_df = pd.DataFrame(test_rows)
equity_df = pd.DataFrame(equity_rows)

train_metrics_df.to_csv(METRICS_DIR / "training_metrics.csv", index=False)
test_metrics_df.to_csv(METRICS_DIR / "evaluation_metrics.csv", index=False)
equity_df.to_csv(METRICS_DIR / "subgroup_equity.csv", index=False)

display(test_metrics_df)
display(equity_df.head())
threshold_counts

## Section 4 - Save Model Result Payload for Notebook 04

This cell writes model-level metrics and subgroup gaps to a JSON payload.

Notebook 04 uses this file to build the five-view Evidence Dashboard.

Do not treat this payload as a final submission file.

In [ ]:
payload = {
    "model_results": model_results,
    "threshold_counts": threshold_counts,
}
model_results_path = METRICS_DIR / "model_results.json"
model_results_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Saved:", model_results_path)
print("Run Notebook 04 next to build the five required evidence views.")
print("Reminder: participants must still complete root reference JSON deliverables manually.")